In [2]:
import json
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from pathlib import Path
from neurodent import core

# 1. Define paths and create directory
data_path = Path("/mnt/isilon/marsh_single_unit/YY_PyEEG/neurodent_Yastika/results/fdsars")
output_dir = data_path / 'summary_output'
hist_dir = output_dir / 'histogram_ISI'

# Create only the histogram directory
hist_dir.mkdir(parents=True, exist_ok=True)

output_csv_path = output_dir / 'spike_summary.csv'
folders = [p for p in data_path.glob("*/*") if p.is_dir()]

all_recordings_data = []

for f_sel in folders:
    fif_search = list(f_sel.rglob("*raw.fif"))
    if not fif_search:
        continue
        
    fif_path = fif_search[0]
    json_name = fif_path.name.replace('-raw.fif', '.json')
    json_path = fif_path.with_name(json_name)   

    # Initialize identifiers
    animal_id, full_genotype, animal_day = "Unknown", "Unknown", "Unknown"
    sex, genotype_status = "Unknown", "Unknown"

    if json_path.is_file():
        with open(json_path, 'r') as file:
            eeg_metadata = json.load(file)
            animal_id = eeg_metadata.get("animal_id", "Unknown")
            full_genotype = eeg_metadata.get("genotype", "Unknown")
            animal_day = eeg_metadata.get("animal_day", "Unknown")
            
            if full_genotype != "Unknown" and len(full_genotype) > 1:
                first_char = full_genotype[0].upper()
                if first_char == 'M':
                    sex, genotype_status = "Male", full_genotype[1:]
                elif first_char == 'F':
                    sex, genotype_status = "Female", full_genotype[1:]
                else:
                    genotype_status = full_genotype 
            else:
                genotype_status = full_genotype
    
    raw = mne.io.read_raw_fif(fif_path, preload=False)
    total_duration = raw.n_times / raw.info['sfreq']
    all_events, all_event_id = mne.events_from_annotations(raw)
    recording_name = f"{animal_id}_{genotype_status}_{animal_day}"
    
    # 2. Initialize 3x4 Subplot grid without constrained_layout for manual control
    fig, axes = plt.subplots(3, 4, figsize=(20, 15))
    axes_flat = axes.flatten()

    row_data = {
        "animal_id": animal_id,
        "sex": sex,
        "genotype": genotype_status,
        "animal_day": animal_day,
        "total_duration_sec": total_duration
    }

    # Loop through the first 10 channels
    for ch_idx, ch_name in enumerate(raw.ch_names):
        if ch_idx >= 10: 
            break
        
        label = f"Spike_Ch{ch_idx}"
        ch_name_abbrev = core.parse_chname_to_abbrev(ch_name)
        # Add channel number in parenthesis to histogram title
        display_name = f"{ch_name_abbrev} ({ch_idx + 1})"
        ax = axes_flat[ch_idx]
        
        if label not in all_event_id:
            row_data[f"{ch_name_abbrev}_spike_count"] = 0
            row_data[f"{ch_name_abbrev}_spike_rate"] = 0.0
            row_data[f"{ch_name_abbrev}_mean_isi"] = np.nan
            ax.set_title(f"{display_name}\nNo Spikes")
            continue
            
        ch_event_id = all_event_id[label]
        ch_events = all_events[all_events[:, 2] == ch_event_id]
        spike_times = ch_events[:, 0] / raw.info['sfreq'] 
        
        spike_count = len(spike_times)
        spike_rate = spike_count / total_duration if total_duration > 0 else 0.0
        
        if spike_count > 1:
            isi = np.diff(spike_times)
            median_isi = np.median(isi)
            bins = np.logspace(np.log10(max(isi.min(), 0.001)), np.log10(isi.max()), 50)
            
            ax.hist(isi, bins=bins, color='skyblue', edgecolor='black', alpha=0.7)
            ax.set_xscale('log')
            ax.axvline(median_isi, color='red', linestyle='--', linewidth=1.5, label=f'Med: {median_isi:.2f}s')
            
            ax.set_title(f"{display_name}\nCnt: {spike_count}, Rate: {spike_rate:.3e} Hz", fontsize=10)
            ax.set_xlabel("ISI (s) (Log Scale)", fontsize=8)
            ax.legend(fontsize=7)
            row_data[f"{ch_name_abbrev}_mean_isi"] = np.mean(isi)
        else:
            ax.set_title(f"{display_name}\n{spike_count} Spike (No ISI)")
            row_data[f"{ch_name_abbrev}_mean_isi"] = np.nan
            
        row_data[f"{ch_name_abbrev}_spike_count"] = spike_count
        row_data[f"{ch_name_abbrev}_spike_rate"] = spike_rate

    # --- 3. Consolidate Event Plot and Add Margin ---
    for i in [10, 11]:
        axes_flat[i].remove()
    
    gs = axes[0, 0].get_gridspec()
    ax_events = fig.add_subplot(gs[2, 2:]) 
    
    if len(all_events) > 0:
        mne.viz.plot_events(all_events, sfreq=raw.info['sfreq'], 
                            event_id=all_event_id, axes=ax_events, show=False)
        
        if ax_events.get_legend():
            ax_events.get_legend().remove() 
            
        for collection in ax_events.collections:
            collection.set_sizes([10])
            collection.set_alpha(0.7)

        ax_events.set_title("Temporal Event Distribution", fontsize=12, pad=15)
        ax_events.tick_params(axis='both', labelsize=8)
        ax_events.set_xlabel("Time (s)", fontsize=10)
    else:
        ax_events.text(0.5, 0.5, "No Events to Plot", ha='center')

    # Add vertical margin (hspace) between rows and extra space for the title
    plt.subplots_adjust(hspace=0.4, wspace=0.3, top=0.92, bottom=0.08)

    fig.suptitle(f"Summary: {recording_name}", fontsize=18)
    fig.savefig(hist_dir / f"{recording_name}_combined_summary.png")
    plt.close(fig)
        
    all_recordings_data.append(row_data)

# 4. Save CSV
df_results = pd.DataFrame(all_recordings_data)
df_results.to_csv(output_csv_path, index=False)

print(f"Summary generated successfully in {hist_dir}")

Opening raw data file /mnt/isilon/marsh_single_unit/YY_PyEEG/neurodent_Yastika/results/fdsars/012022_cohort4_group5_3mice__fwt_mmut_fmut-fmut/FMUT-FMut-FMUT FMut Jan-22-2022/FMUT-FMut-FMUT FMut Jan-22-2022-raw.fif...
    Reading extended channel information
Isotrak not found
    Range : 0 ... 53627999 =      0.000 ... 53627.999 secs
Ready.
Opening raw data file /mnt/isilon/marsh_single_unit/YY_PyEEG/neurodent_Yastika/results/fdsars/012022_cohort4_group5_3mice__fwt_mmut_fmut-fmut/FMUT-FMut-FMUT FMut Jan-22-2022/FMUT-FMut-FMUT FMut Jan-22-2022-raw-1.fif...
    Reading extended channel information
Isotrak not found
    Range : 53628000 ... 57604319 =  53628.000 ... 57604.319 secs
Ready.
Used Annotations descriptions: [np.str_('Spike_Ch0'), np.str_('Spike_Ch1'), np.str_('Spike_Ch2'), np.str_('Spike_Ch3'), np.str_('Spike_Ch4'), np.str_('Spike_Ch5'), np.str_('Spike_Ch6'), np.str_('Spike_Ch7'), np.str_('Spike_Ch8'), np.str_('Spike_Ch9')]
Opening raw data file /mnt/isilon/marsh_single_unit/YY_

: 

In [ ]:
import json
import numpy as np
import pandas as pd
import mne
from pathlib import Path
from neurodent import core

# 1. Define your input and output paths
data_path = Path("/mnt/isilon/marsh_single_unit/YY_PyEEG/neurodent_Yastika/results/fdsars")
output_csv_path = data_path / 'spike_summary.csv'

folders = [p for p in data_path.glob("*/*") if p.is_dir()]

# This list will hold a dictionary for each animal recording
all_recordings_data = []

for f_sel in folders:
    fif_search = list(f_sel.rglob("*raw.fif"))
    
    if not fif_search:
        continue
        
    fif_path = fif_search[0]
    json_name = fif_path.name.replace('-raw.fif', '.json')
    json_path = fif_path.with_name(json_name)   

    # Initialize default identifiers
    animal_id, full_genotype, animal_day = "Unknown", "Unknown", "Unknown"
    sex, genotype_status = "Unknown", "Unknown"

    if json_path.is_file():
        with open(json_path, 'r') as file:
            eeg_metadata = json.load(file)
            animal_id = eeg_metadata.get("animal_id", "Unknown")
            full_genotype = eeg_metadata.get("genotype", "Unknown")
            animal_day = eeg_metadata.get("animal_day", "Unknown")
            
            # --- New Logic: Parse Sex and Genotype ---
            if full_genotype != "Unknown" and len(full_genotype) > 1:
                first_char = full_genotype[0].upper()
                if first_char == 'M':
                    sex = "Male"
                    genotype_status = full_genotype[1:]
                elif first_char == 'F':
                    sex = "Female"
                    genotype_status = full_genotype[1:]
                else:
                    # If the first char isn't M/F, keep original
                    genotype_status = full_genotype 
            else:
                genotype_status = full_genotype
            # -----------------------------------------
    else:
        print(f"Warning: No matching JSON file found for {fif_path.name}")
        
    raw = mne.io.read_raw_fif(fif_path, preload=False)
    total_duration = raw.times[-1]

    # 2. Updated row dictionary with 'Sex' and 'Genotype'
    row_data = {
        "animal_id": animal_id,
        "sex": sex,
        "genotype": genotype_status,
        "animal_day": animal_day,
        "total_duration_sec": total_duration
    }

    all_events, all_event_id = mne.events_from_annotations(raw)

    for ch_idx, ch_name in enumerate(raw.ch_names):
        label = f"Spike_Ch{ch_idx}"
        ch_name_abbrev = core.parse_chname_to_abbrev(ch_name)
        
        if label not in all_event_id:
            row_data[f"{ch_name_abbrev}_spike_count"] = 0
            row_data[f"{ch_name_abbrev}_spike_rate"] = 0.0
            row_data[f"{ch_name_abbrev}_mean_isi"] = np.nan
            continue
            
        ch_event_id = all_event_id[label]
        ch_events = all_events[all_events[:, 2] == ch_event_id]
        
        spike_samples = ch_events[:, 0]
        spike_times = spike_samples / raw.info['sfreq'] 
        
        spike_count = len(spike_times)
        spike_rate = spike_count / total_duration if total_duration > 0 else 0.0
        
        if spike_count > 1:
            isi = np.diff(spike_times)
            median_isi = np.median(isi)
        else:
            median_isi = np.nan
            
        row_data[f"{ch_name_abbrev}_spike_count"] = spike_count
        row_data[f"{ch_name_abbrev}_spike_rate"] = spike_rate
        row_data[f"{ch_name_abbrev}_mean_isi"] = median_isi
        
    all_recordings_data.append(row_data)

df_results = pd.DataFrame(all_recordings_data)
df_results.to_csv(output_csv_path, index=False)

print(f"Successfully saved {len(all_recordings_data)} recordings to {output_csv_path}")

In [ ]:
import mne
from pathlib import Path
import json
import numpy as np

In [ ]:
folders = [p for p in data_path.glob("*/*") if p.is_dir()]

folder_sel = folders[0]
fif_search = list(Path(folder_sel).rglob("*raw.fif"))

fif_path = fif_search[0]

json_path = fif_path.name.replace('-raw.fif', '.json')

print(json_path)

# json_path = fif_path.replace('-raw.fif', '.json')
# json_pah

In [ ]:
# Get list of all animal/dates folder

data_path = Path("/mnt/isilon/marsh_single_unit/YY_PyEEG/neurodent_Yastika/results/fdsars")

# The "*/*" pattern strictly matches items exactly two levels deep.
# The .is_dir() method ensures only directories are included in the list.
folders = [p for p in data_path.glob("*/*") if p.is_dir()]

for f_sel in folders:
    fif_search = list(Path(folder_sel).rglob("*raw.fif"))
    fif_path = fif_search[0]


    json_name = fif_path.name.replace('-raw.fif', '.json')
    json_path = fif_path.with_name(json_name)   

    if json_path.is_file():
        # Open and load the JSON data into a Python dictionary
        with open(json_path, 'r') as file:
            eeg_metadata = json.load(file)
            print(f"Successfully read JSON data: {eeg_metadata.keys()}")
    else:
        print(f"Warning: No matching JSON file found for {fif_file.name}")
        eeg_metadata = {}
        
    # 4. Load your EEG data
    raw = mne.io.read_raw_fif(fif_path)

    # Compute spike interval and spike frequency per channel
    results = {}
    total_duration = raw.times[-1]
    for ch_idx, ch_name in enumerate(raw.ch_names):
        label = f"Spike_Ch{ch_idx}"
        
        if label not in all_event_id:
            results[ch_name] = {
                "mean_isi": np.nan,
                "spike_count": 0,
                "spike_rate": 0.0,
            }
            continue
        # Get spike events for this channel
        ch_event_id = all_event_id[label]
        ch_events = all_events[all_events[:, 2] == ch_event_id]
        spike_count = len(spike_times)


In [ ]:

import mne
from pathlib import Path

# Assuming 'files_to_process' is the list generated from the previous function
for fif_file in files_to_process:
    print(f"Loading primary file: {fif_file.name}")
    
    # 1. Get the string name and replace '-raw.fif' with '.json'
    json_name = fif_file.name.replace('-raw.fif', '.json')
    
    # 2. Construct the full path to the JSON file in the same directory
    json_path = fif_file.with_name(json_name)
    
    # 3. Check if the JSON file exists before trying to read it
    if json_path.is_file():
        # Open and load the JSON data into a Python dictionary
        with open(json_path, 'r') as file:
            eeg_metadata = json.load(file)
            print(f"Successfully read JSON data: {eeg_metadata.keys()}")
    else:
        print(f"Warning: No matching JSON file found for {fif_file.name}")
        eeg_metadata = {}
        
    # 4. Load your EEG data
    raw = mne.io.read_raw_fif(fif_file, preload=True)
    
    # Now you have both 'raw' (the EEG data) and 'eeg_metadata' (the JSON dict)
    # ready for your analysis pipeline.

In [ ]:
# LATER THIS WILL BE A FOR LOOP

folder_sel = folders[0]

# Find all .fif files in data_path
fif_test = Path(folder_sel).rglob("*raw.fif")
print(fif_list)

# fif_test = fif_list[0]

# print(fif_test)

# raw = mne.io.read_raw_fif(fif_test)
 
# all_events, all_event_id = mne.events_from_annotations(raw)

# mne.viz.plot_events(events=all_events, event_id=all_event_id, sfreq=raw.info["sfreq"])


In [ ]:
# Compute spike interval and spike frequency per channel
results = {}
total_duration = raw.times[-1]
for ch_idx, ch_name in enumerate(raw.ch_names):
    label = f"Spike_Ch{ch_idx}"
    
    if label not in all_event_id:
        results[ch_name] = {
            "mean_isi": np.nan,
            "spike_count": 0,
            "spike_rate": 0.0,
        }
        continue
    # Get spike events for this channel
    ch_event_id = all_event_id[label]
    ch_events = all_events[all_events[:, 2] == ch_event_id]
    spike_count = len(spike_times)
    # Compute inter-spike intervals


In [ ]:
    isi = np.diff(spike_times)
    mean_isi = np.mean(isi) if len(isi) > 0 else np.nan
    # Compute spike frequency
    spike_rate = spike_count / total_duration
    results[ch_name] = {
        "mean_isi": mean_isi,
        "spike_count": spike_count,
        "spike_rate": spike_rate,
    }
# Display results
for ch_name, data in results.items():
    print(f"{ch_name}: {data['spike_count']} spikes, "
          f"mean ISI = {data['mean_isi']:.4f}s, "
          f"rate = {data['spike_rate']:.4f} spikes/s")

# Cross correlogram
 
